# VideoStir MedVidQA Reproduction Notebook

This notebook runs VideoStir from the current `src/videostir` package against MedVidQA metadata and locally mounted videos.

**Prerequisites:**
- Open this notebook from a VideoStir checkout and install the package with the setup cell below.
- Attach the MedVidQA metadata and video datasets in Kaggle. Expected files are `MedVidQA/{train,val,test}.json` and `videos/<YouTube video ID>.mp4`; neither data folder is included in GitHub.
- Set `MEDVIDQA_ROOT` and `VIDEO_ROOT` below if Kaggle mounted the datasets elsewhere.
- Download the Frame Reranker LoRA adapter from [Google Drive](https://drive.google.com/file/d/1CUC1i7zstZktWDp30Pts4s8E7QULDgns/view?usp=drive_link) and extract it to `result/`.
- Enable Kaggle internet access if required model weights are not already cached.

In [ ]:
# Install the src-layout package and locate the attached Kaggle datasets
!pip install -e ".[eval,qwen35]"

from pathlib import Path
import os

MEDVIDQA_ROOT = Path(os.environ.get("MEDVIDQA_ROOT", "MedVidQA"))
VIDEO_ROOT = Path(os.environ.get("VIDEO_ROOT", "videos"))
INTENT_MODEL_ID = "Qwen/Qwen3.5-0.8B"
RERANKER_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
EMBEDDING_MODEL_NAME = "PE-Core-G14-448"
assert (MEDVIDQA_ROOT / "test.json").is_file(), f"Set MEDVIDQA_ROOT to the folder containing test.json: {MEDVIDQA_ROOT}"
assert VIDEO_ROOT.is_dir(), f"Set VIDEO_ROOT to the folder containing video files: {VIDEO_ROOT}"
print(f"MedVidQA: {MEDVIDQA_ROOT.resolve()}")
print(f"Videos: {VIDEO_ROOT.resolve()} ({len(list(VIDEO_ROOT.glob('*.mp4')))} mp4 files)")

In [ ]:
# Verify installation and check GPU availability
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")

## 1. Select a MedVidQA Test Sample

Use test item 2711, whose video is `h5MvX50zTLM.mp4` and answer interval is 7–73 seconds.

In [ ]:
import json

with (MEDVIDQA_ROOT / "test.json").open() as f:
    test_data = json.load(f)
sample = next(s for s in test_data if s["sample_id"] == 2711)
video_path = VIDEO_ROOT / f"{sample['video_id']}.mp4"
assert video_path.is_file(), f"Missing local video: {video_path}"
print(f"Sample {sample['sample_id']} | {sample['video_id']}")
print(sample["question"])
print(f"Answer interval: {sample['answer_start_second']}–{sample['answer_end_second']} s")
print(f"Video: {video_path}")

## 2. Run One Retrieval Sample

Start with this single local sample. The dataset-wide CLI uses the default 7B intent model; on a memory-limited GPU, the smaller model below is a better first run. The Frame Reranker LoRA adapter must exist under `result/`.

In [ ]:
from videostir.inference import PipelineConfig, run_pipeline

config = PipelineConfig(
    video_path=str(video_path),
    query=sample["question"],
    output_dir="/kaggle/working/videostir-results",
    intent_model_id=INTENT_MODEL_ID,
    reranker_model_id=RERANKER_MODEL_ID,
    embedding_model_name=EMBEDDING_MODEL_NAME,
    reranker_adapter_dir="result",
    top_frames=32,
    batch_size=4,
    enable_checkpoint=True,
    checkpoint_dir="/kaggle/working/videostir-checkpoints",
    cache_dir="/kaggle/working/videostir-cache",
)

result = run_pipeline(config)
print(f"Reranked frames: {len(result.reranked_frames)}")
print(f"Time-focused frames: {len(result.time_focus_frames)}")
print(f"Intent: {result.intent}")

## 3. Evaluate This Sample

Compare the retrieved frame timestamps with this sample’s annotated answer interval.

In [ ]:
from videostir.eval import MedVidQAEvaluator

evaluator = MedVidQAEvaluator(medvidqa_root=str(MEDVIDQA_ROOT))
sample_metrics = evaluator.evaluate_sample(
    sample={
        "sample_id": sample["sample_id"],
        "video_id": sample["video_id"],
        "question": sample["question"],
        "answer": sample.get("answer", ""),
        "answer_start_second": sample["answer_start_second"],
        "answer_end_second": sample["answer_end_second"],
        "video_length": sample.get("video_length", 0),
    },
    retrieved_frames=result.reranked_frames,
)

for key in ["mrr", "recall@10", "recall@25", "recall@50", "recall@128", "map@10", "coverage@128"]:
    print(f"{key:15s}: {sample_metrics[key]:.4f}")

## 4. Optional Batch Processing

Use this only after the single-sample run works. It uses the same mounted video path.

In [ ]:
batch_config = [{
    "id": str(sample["sample_id"]),
    "video_path": str(video_path),
    "query": sample["question"],
}]

with open("batch_config.json", "w") as f:
    json.dump(batch_config, f, indent=2)

# from videostir.inference import run_batch_from_config
# run_batch_from_config("batch_config.json", "/kaggle/working/batch-results")

## 5. CLI Reference

```bash
# Single video retrieval (uses the default 7B intent model)
python -m videostir.inference run --video videos/h5MvX50zTLM.mp4 \
  --query "How to perform chin tucks to treat neck pain?" --output results/ \
  --intent-model-id Qwen/Qwen3.5-0.8B \
  --reranker-model-id Qwen/Qwen2.5-VL-3B-Instruct \
  --embedding-model-name PE-Core-G14-448 \
  --reranker-adapter-dir result

# Dataset information and dataset retrieval
python -m videostir.eval info --medvidqa-root MedVidQA
python -m videostir.eval retrieve --dataset test --num-samples 1 \
  --medvidqa-root MedVidQA --video-root videos --output-dir medvidqa_retrieval
```